In [1]:
%load_ext autoreload
%autoreload 2

# Recoil heating comparison

We consider a prolate silica spheroid centered at the origin, with its long axis aligned with the $x$ axis. The particle is illuminated by a monochromatic plane wave propagating along $+z$ and linearly polarized along $x$. We compare the analytical, orientation-derivative, and angular-spectrum calculations using the same particle and illumination parameters.

In [2]:
from IPython.display import display, Markdown
import numpy as np
from scipy.constants import epsilon_0, c, hbar
import optomechanics_levitation as oml

In [3]:
## PARTICLE PROPERTIES
long_axis = 100.0; long_radius = long_axis / 2.0; aspect_ratio = 2.0; short_radius = long_radius / aspect_ratio # [nm]
wavelength = 1550.0 # [nm]
intensity = 1.0e9 # [W/m^2]

# Discretization step
if aspect_ratio == 2.0:
    mesh_divisions = 35.0
else:
    mesh_divisions = 65.0
step = long_axis / mesh_divisions # [nm]

# x-oriented ellipsoidal particle
ellipsoidal_particle = oml.particle_properties(particle_type="spheroid", material_name="sio2", step_nm=step, mesh="cube",
                                               center=True, R1=long_radius / step - 0.2, R2=short_radius / step - 0.2, R3=short_radius / step - 0.2)

# INTERACTION PARAMETERS
environment = {"eps1": 1.0, "eps2": 1.0, "eps3": 1.0, "spacing": 0.0}
far_field_radius = 1e6 # [nm]
n_theta = 64; n_phi = 64

## Theory approach

In [4]:
epsilon_r = ellipsoidal_particle["material"].epsilon(wavelength).real
theory_result = oml.TheoreticalTorqueRecoilHeating(long_radius, short_radius, epsilon_r, wavelength_nm=wavelength, intensity_w_per_m2=intensity).compute()
s_theory = theory_result["total_torque"]; d_theory = theory_result["anisotropy_term"]
gamma_theory = 2 * np.pi * s_theory / (hbar**2 * d_theory)

## Derivative approach

In [5]:
plane_wave = oml.PlaneWave(direction=(0.0, 0.0, 1.0), polarization_state=(1.0, 0.0))
amplitude_v_per_m = np.sqrt(2 * intensity / (epsilon_0 * c))
derivative_result = oml.DerivativeTorqueRecoilHeating(ellipsoidal_particle, wavelength, environment,
                                                       far_field_distance_nm=far_field_radius, rotation_axes=("y", "z"),
                                                       angle_rad=0.005, n_theta=n_theta, n_phi=n_phi).compute(
                                                           plane_wave, amplitude_v_per_m=amplitude_v_per_m)
d_derivative = derivative_result["D"]; s_derivative = derivative_result["total_torque"]
gamma_derivative = 2 * np.pi * s_derivative["y"] / (hbar**2 * d_derivative["y"])

## Angular-spectrum approach

In [6]:
spectral_calculator = oml.SpectralTorqueRecoilHeating(ellipsoidal_particle, wavelength, environment, torque_axis="y",
                                                        n_theta=n_theta, n_phi=n_phi, radius_wavelengths=far_field_radius / wavelength)
spectral_results = {}
for axis in ("y", "z"):
    spectral_calculator.torque_axis = axis
    spectral_results[axis] = spectral_calculator.compute(plane_wave, amplitude_v_per_m=amplitude_v_per_m)
s_spectral = {axis: spectral_results[axis]["total_torque"] for axis in ("y", "z")}

# Results

The table compares the three predictions for $S_{\tau_y\tau_y}$ and $S_{\tau_z\tau_z}$. The analytical and orientation-derivative methods also provide the scattering rate $\Gamma_{\mathrm{sc}}$ and dimensionless response $D_j$. Since their classes return $S_{\tau\tau}$ and $D_j$, we recover $\Gamma_{\mathrm{sc}}=2\pi S_{\tau\tau}/(\hbar^2D_j)$ for the table. The spectral method returns the total torque noise and its spin, orbital, and correlation terms; it does not report $\Gamma_{\mathrm{sc}}$ or $D_j$. Relative differences use the analytical prediction as the reference.

In [7]:
gamma_diff = 100 * np.abs(gamma_derivative - gamma_theory) / gamma_theory
s_diff_derivative = {axis: 100 * np.abs(s_derivative[axis] - s_theory) / s_theory for axis in ("y", "z")}
s_diff_spectral = {axis: 100 * np.abs(s_spectral[axis] - s_theory) / s_theory for axis in ("y", "z")}
d_diff = {axis: 100 * np.abs(d_derivative[axis] - d_theory) / d_theory for axis in ("y", "z")}

In [8]:
comparison_table = rf"""

| Observable | Theory | Derivative | Spectral | Derivative difference | Spectral difference |
|:-:|:-:|:-:|:-:|:-:|:-:|
| $\Gamma_{{\mathrm{{sc}}}}\,[\mathrm{{s}}^{{-1}}]$ | {gamma_theory:.3e} | {gamma_derivative:.3e} | — | {gamma_diff:.2f}% | — |
| $S_{{\tau_y\tau_y}}\,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$ | {s_theory:.3e} | {s_derivative['y']:.3e} | {s_spectral['y']:.3e} | {s_diff_derivative['y']:.2f}% | {s_diff_spectral['y']:.2f}% |
| $S_{{\tau_z\tau_z}}\,[\mathrm{{N}}^{{2}}\mathrm{{m}}^{{2}}/\mathrm{{Hz}}]$ | {s_theory:.3e} | {s_derivative['z']:.3e} | {s_spectral['z']:.3e} | {s_diff_derivative['z']:.2f}% | {s_diff_spectral['z']:.2f}% |
| $D_y$ | {d_theory:.6f} | {d_derivative['y']:.6f} | — | {d_diff['y']:.2f}% | — |
| $D_z$ | {d_theory:.6f} | {d_derivative['z']:.6f} | — | {d_diff['z']:.2f}% | — |

"""

display(Markdown(comparison_table))



| Observable | Theory | Derivative | Spectral | Derivative difference | Spectral difference |
|:-:|:-:|:-:|:-:|:-:|:-:|
| $\Gamma_{\mathrm{sc}}\,[\mathrm{s}^{-1}]$ | 1.597e+09 | 1.613e+09 | — | 1.00% | — |
| $S_{\tau_y\tau_y}\,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$ | 9.114e-62 | 9.165e-62 | 9.166e-62 | 0.56% | 0.58% |
| $S_{\tau_z\tau_z}\,[\mathrm{N}^{2}\mathrm{m}^{2}/\mathrm{Hz}]$ | 9.114e-62 | 9.167e-62 | 9.167e-62 | 0.58% | 0.59% |
| $D_y$ | 0.032240 | 0.032099 | — | 0.44% | — |
| $D_z$ | 0.032240 | 0.032108 | — | 0.41% | — |



# Notes

**Mesh resolution.** The cubic mesh uses a step size $s=L/N$, where $L$ is the long-axis length and $N$ is `mesh_divisions`. We tested several values of $N$ for the 2:1 spheroid. Refining the mesh improved the representation of the particle, but also increased computation time substantially. We use $N=35$ for this comparison. For a 4:1 spheroid, $N=65$ keeps a similar resolution across the shorter radius.

**Spheroid boundary.** The `pyGDM2.structures.spheroid` generator adds $0.2$ mesh cells to each input radius when selecting dipoles. We therefore pass $r_i/s-0.2$ for each axis so that its selection criterion uses the intended radii. This offset is fixed in mesh cells and does not change with the aspect ratio.

For the 100 nm, 2:1 particle at $N=35$, the difference between the mesh volume, $N_{\mathrm{dipoles}}s^3$, and the target ellipsoid volume decreases from **+4.82% to +0.26%** after this correction.